# Notebook 04 — Curriculum and Topic Mapping

## Principle
Mapping confidence is measured against a manual gold set.
We do not claim high confidence until agreement is quantified.

## This notebook phase
1. Paths and inputs
2. Taxonomy v1
3. Load question structure
4. Create 2025 P1 gold-set template
5. Save gold set for manual labelling

Mapping rules come after the gold set exists.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
META_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"
QUESTIONS_DIR = PROCESSED_DIR / "questions"
TAXONOMY_DIR = PROCESSED_DIR / "taxonomy"
GOLD_DIR = PROCESSED_DIR / "gold_sets"

for d in [QUESTIONS_DIR, TAXONOMY_DIR, GOLD_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

question_structure_path = QUESTIONS_DIR / "question_structure.csv"
taxonomy_path = TAXONOMY_DIR / "taxonomy_math_v1.csv"
gold_template_path = GOLD_DIR / "gold_2025_p1_template.csv"
gold_labelled_path = GOLD_DIR / "gold_2025_p1_labelled.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Questions:", question_structure_path.exists())

PROJECT_ROOT: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
Questions: True


In [2]:
taxonomy_rows = [
    # topic, subtopic, skill, caps_section, notes
    ("Algebra & Equations", "Linear equations", "Solve linear equations", "Algebra", "v1"),
    ("Algebra & Equations", "Quadratic equations", "Solve quadratics by factorisation", "Algebra", "v1"),
    ("Algebra & Equations", "Quadratic equations", "Solve quadratics by formula", "Algebra", "v1"),
    ("Algebra & Equations", "Inequalities", "Solve and represent inequalities", "Algebra", "v1"),
    ("Algebra & Equations", "Algebraic manipulation", "Simplify/factorise/expand expressions", "Algebra", "v1"),
    ("Algebra & Equations", "Exponents & surds", "Simplify exponential and surd expressions", "Algebra", "v1"),

    ("Number Patterns & Sequences", "Arithmetic sequences", "Find nth term / sum of arithmetic sequence", "Patterns", "v1"),
    ("Number Patterns & Sequences", "Geometric sequences", "Find nth term / sum of geometric sequence", "Patterns", "v1"),
    ("Number Patterns & Sequences", "Quadratic patterns", "Determine quadratic pattern rules", "Patterns", "v1"),

    ("Functions & Graphs", "Linear functions", "Interpret or sketch linear functions", "Functions", "v1"),
    ("Functions & Graphs", "Quadratic functions", "Interpret/sketch parabola and parameters", "Functions", "v1"),
    ("Functions & Graphs", "Hyperbola", "Interpret/sketch hyperbola", "Functions", "v1"),
    ("Functions & Graphs", "Exponential functions", "Interpret/sketch exponential functions", "Functions", "v1"),
    ("Functions & Graphs", "Inverse functions", "Determine or interpret inverses", "Functions", "v1"),

    ("Finance", "Compound interest", "Apply compound growth formulae", "Finance", "v1"),
    ("Finance", "Decay / reduction", "Apply reduction formulae", "Finance", "v1"),
    ("Finance", "Annuities", "Present/future value calculations", "Finance", "v1"),

    ("Calculus", "First principles", "Differentiate from first principles", "Calculus", "v1"),
    ("Calculus", "Rules of differentiation", "Apply differentiation rules", "Calculus", "v1"),
    ("Calculus", "Cubic graphs", "Analyse cubic using calculus", "Calculus", "v1"),
    ("Calculus", "Optimisation", "Solve optimisation problems", "Calculus", "v1"),

    ("Probability", "Basic probability", "Compute simple probabilities", "Probability", "v1"),
    ("Probability", "Venn / mutually exclusive", "Use set relationships in probability", "Probability", "v1"),
    ("Probability", "Tree diagrams / counting", "Use tree diagrams or counting principles", "Probability", "v1"),

    ("Trigonometry", "Identities", "Prove or apply trig identities", "Trigonometry", "v1"),
    ("Trigonometry", "Equations", "Solve trigonometric equations", "Trigonometry", "v1"),
    ("Trigonometry", "2D/3D applications", "Solve trig application problems", "Trigonometry", "v1"),
    ("Trigonometry", "Graphs", "Interpret/sketch trig graphs", "Trigonometry", "v1"),

    ("Euclidean Geometry", "Circle geometry", "Apply circle theorems", "Geometry", "v1"),
    ("Euclidean Geometry", "Similarity / proportion", "Use similarity and proportion", "Geometry", "v1"),
    ("Euclidean Geometry", "Riders / proofs", "Complete geometric riders/proofs", "Geometry", "v1"),

    ("Analytical Geometry", "Distance / midpoint / gradient", "Apply analytic geometry formulae", "Analytical Geometry", "v1"),
    ("Analytical Geometry", "Equation of a line", "Determine equation of a line", "Analytical Geometry", "v1"),
    ("Analytical Geometry", "Circles", "Equation/properties of a circle", "Analytical Geometry", "v1"),

    ("Statistics", "Regression / correlation", "Interpret regression and correlation", "Statistics", "v1"),
    ("Statistics", "Representations", "Interpret statistical representations", "Statistics", "v1"),

    ("Mixed / Multi-topic", "Mixed", "Multi-topic assessed skill", "Mixed", "v1"),
    ("Unmapped", "Unmapped", "Insufficient evidence to map", "Unmapped", "v1"),
]

taxonomy_df = pd.DataFrame(
    taxonomy_rows,
    columns=["topic", "subtopic", "skill", "caps_section", "taxonomy_version"]
)

taxonomy_df.to_csv(taxonomy_path, index=False)
print("Taxonomy rows:", len(taxonomy_df))
print("Topics:", taxonomy_df["topic"].nunique())
display(taxonomy_df.head(12))

Taxonomy rows: 38
Topics: 12


,topic,subtopic,skill,caps_section,taxonomy_version
0,Algebra & Equations,Linear equations,Solve linear equations,Algebra,v1
1,Algebra & Equations,Quadratic equations,Solve quadratics by factorisation,Algebra,v1
2,Algebra & Equations,Quadratic equations,Solve quadratics by formula,Algebra,v1
3,Algebra & Equations,Inequalities,Solve and represent inequalities,Algebra,v1
4,Algebra & Equations,Algebraic manipulation,Simplify/factorise/expand expressions,Algebra,v1
5,Algebra & Equations,Exponents & surds,Simplify exponential and surd expressions,Algebra,v1
6,Number Patterns & Sequences,Arithmetic sequences,Find nth term / sum of arithmetic sequence,Patterns,v1
7,Number Patterns & Sequences,Geometric sequences,Find nth term / sum of geometric sequence,Patterns,v1
8,Number Patterns & Sequences,Quadratic patterns,Determine quadratic pattern rules,Patterns,v1
9,Functions & Graphs,Linear functions,Interpret or sketch linear functions,Functions,v1


In [3]:
assert taxonomy_df["topic"].isna().sum() == 0
assert taxonomy_df["subtopic"].isna().sum() == 0
assert taxonomy_df.duplicated(["topic", "subtopic", "skill"]).sum() == 0

print("Taxonomy validation passed")
print(taxonomy_df["topic"].value_counts())

Taxonomy validation passed
topic
Algebra & Equations            6
Functions & Graphs             5
Calculus                       4
Trigonometry                   4
Number Patterns & Sequences    3
Finance                        3
Probability                    3
Euclidean Geometry             3
Analytical Geometry            3
Statistics                     2
Mixed / Multi-topic            1
Unmapped                       1
Name: count, dtype: int64


In [4]:
questions_df = pd.read_csv(question_structure_path)

# Stable question_id
def make_question_id(row):
    qn = row.get("question_number")
    subq = row.get("subquestion")
    qn_part = f"Q{int(qn)}" if pd.notna(qn) else "QNA"
    sub_part = str(subq) if pd.notna(subq) else "main"
    return f"{row['document_id']}__{qn_part}__{sub_part}"

questions_df["question_id"] = questions_df.apply(make_question_id, axis=1)

print("Question records:", len(questions_df))
print("Unique question_id:", questions_df["question_id"].nunique())
display(questions_df.head(3))

Question records: 417
Unique question_id: 414


,document_id,year,paper,document_type,source_text_file,extraction_quality,question_number,subquestion,marks,marks_all_found,question_text,question_char_count,segment_index,segmentation_status,question_id
0,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1,3.0,[3],1.1 Solve for x:\n\n11.1 x +x-12=0 (3),36.0,1.0,ok,2023_nov_p1_exam_maths__Q1__1.1
1,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1.2,4.0,[4],1.1.2 3x*-2x=6 (answers correct to TWO decimal...,58.0,2.0,ok,2023_nov_p1_exam_maths__Q1__1.1.2
2,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1.3,4.0,[4],1.1.3 V2xt+1 =x-1 (4),21.0,3.0,ok,2023_nov_p1_exam_maths__Q1__1.1.3


In [5]:
gold_source = questions_df[
    (questions_df["document_id"] == "2025_nov_p1_exam_maths")
    & (questions_df["segmentation_status"] == "ok")
].copy()

print("2025 P1 candidate rows:", len(gold_source))
display(
    gold_source[
        ["question_id", "question_number", "subquestion", "marks", "question_text"]
    ].head(15)
)

2025 P1 candidate rows: 50


,question_id,question_number,subquestion,marks,question_text
260,2025_nov_p1_exam_maths__Q1__1.1.2,1.0,1.1.2,NaN,1.1.2 5x? +2=-9x (correct to TWO decimal places)
261,2025_nov_p1_exam_maths__Q1__1.1.3,1.0,1.1.3,NaN,1.1.3 8x? > 2x
262,2025_nov_p1_exam_maths__Q1__1.1.4,1.0,1.1.4,NaN,1.1.4 2.27* -9.2* +4=0\n\n[ft 1
263,2025_nov_p1_exam_maths__Q1__1.1.5,1.0,1.1.5,NaN,1.1.5 —+2=—=\nx Vx
264,2025_nov_p1_exam_maths__Q1__1.2,1.0,1.2,NaN,1.2 Calculate the values of x and y if:\n\ne x...
265,2025_nov_p1_exam_maths__Q2__2.1,2.0,2.1,NaN,2.1 Given the infinite geometric series: (f+ 1...
266,2025_nov_p1_exam_maths__Q2__2.1.1,2.0,2.1.1,NaN,2.1.1 Show that t=—2
267,2025_nov_p1_exam_maths__Q2__2.1.2,2.0,2.1.2,NaN,"2.1.2 Calculate the value of 7,,. Write your a..."
268,2025_nov_p1_exam_maths__Q2__2.1.3,2.0,2.1.3,NaN,2.1.3 Calculate the sum of the infinite series...
269,2025_nov_p1_exam_maths__Q2__2.2,2.0,2.2,NaN,2.2 Given }'(4p-1)= 26 675\npak


In [6]:
gold_template = gold_source[
    [
        "question_id",
        "document_id",
        "year",
        "paper",
        "question_number",
        "subquestion",
        "marks",
        "question_text",
    ]
].copy()

# Manual label columns
gold_template["topic_gold"] = ""
gold_template["subtopic_gold"] = ""
gold_template["skill_gold"] = ""
gold_template["secondary_topic_gold"] = ""
gold_template["command_verb_gold"] = ""
gold_template["structure_type_gold"] = ""
gold_template["notes"] = ""
gold_template["labeler"] = ""
gold_template["labelled_at"] = ""

gold_template.to_csv(gold_template_path, index=False)
print("Gold template saved:", gold_template_path)
print("Rows to label:", len(gold_template))

Gold template saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\gold_sets\gold_2025_p1_template.csv
Rows to label: 50


## Gold-set labelling instructions

Manually label: `data/processed/gold_sets/gold_2025_p1_template.csv`

For each row fill:
- `topic_gold` from taxonomy topics
- `subtopic_gold` from taxonomy subtopics
- `skill_gold` short assessed skill
- `secondary_topic_gold` only if needed
- `command_verb_gold` (calculate/determine/sketch/show/prove/...)
- `structure_type_gold` (routine_calculation/multi_step/interpretation/proof/...)

Rules:
1. Label the **assessed** skill, not every prerequisite
2. Use taxonomy topics only
3. If unsure, use `Unmapped` and explain in notes
4. Save completed file as:
   `data/processed/gold_sets/gold_2025_p1_labelled.csv`

In [7]:
if gold_labelled_path.exists():
    gold_df = pd.read_csv(gold_labelled_path)
    print("Labelled gold set loaded:", len(gold_df))
    print("Topic coverage:")
    print(gold_df["topic_gold"].value_counts(dropna=False))
else:
    print("Waiting for manual labels at:")
    print(gold_labelled_path)

Waiting for manual labels at:
c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\gold_sets\gold_2025_p1_labelled.csv


In [8]:
from pathlib import Path
import re
import json
from datetime import datetime, timezone

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
QUESTIONS_DIR = PROCESSED_DIR / "questions"
TAXONOMY_DIR = PROCESSED_DIR / "taxonomy"
GOLD_DIR = PROCESSED_DIR / "gold_sets"
MAPPED_DIR = PROCESSED_DIR / "mapped"
MAPPED_DIR.mkdir(parents=True, exist_ok=True)

gold_path = GOLD_DIR / "gold_2025_p1_labelled.csv"
question_structure_path = QUESTIONS_DIR / "question_structure.csv"
taxonomy_path = TAXONOMY_DIR / "taxonomy_math_v1.csv"

gold_df = pd.read_csv(gold_path)
questions_df = pd.read_csv(question_structure_path)

print("Gold rows:", len(gold_df))
print("Question structure rows:", len(questions_df))
print("\nGold topic distribution:")
print(gold_df["topic_gold"].value_counts())

assert len(gold_df) > 0, "Gold set is empty"
assert gold_df["topic_gold"].isna().sum() == 0, "Missing topic_gold labels"
print("\nGold set validation passed")

Gold rows: 49
Question structure rows: 417

Gold topic distribution:
topic_gold
Functions & Graphs             15
Calculus                       10
Number Patterns & Sequences     9
Algebra & Equations             6
Probability                     5
Finance                         4
Name: count, dtype: int64

Gold set validation passed


In [9]:
taxonomy_rows = [
    ("Algebra & Equations", "Linear equations", "Solve linear equations", "Algebra", "v1.1"),
    ("Algebra & Equations", "Quadratic equations", "Solve quadratic equations", "Algebra", "v1.1"),
    ("Algebra & Equations", "Inequalities", "Solve inequalities", "Algebra", "v1.1"),
    ("Algebra & Equations", "Exponents & surds", "Simplify/solve exponential and surd expressions", "Algebra", "v1.1"),
    ("Algebra & Equations", "Algebraic manipulation", "Simplify/factorise/expand expressions", "Algebra", "v1.1"),
    ("Algebra & Equations", "Simultaneous equations", "Solve simultaneous equations", "Algebra", "v1.1"),

    ("Number Patterns & Sequences", "Arithmetic sequences", "Work with arithmetic sequences/series", "Patterns", "v1.1"),
    ("Number Patterns & Sequences", "Geometric sequences", "Work with geometric sequences/series", "Patterns", "v1.1"),
    ("Number Patterns & Sequences", "Quadratic sequences", "Work with quadratic sequences", "Patterns", "v1.1"),
    ("Number Patterns & Sequences", "Series & sigma notation", "Evaluate series and sigma notation", "Patterns", "v1.1"),

    ("Functions & Graphs", "Linear functions", "Interpret/sketch linear functions", "Functions", "v1.1"),
    ("Functions & Graphs", "Quadratic functions", "Interpret/sketch parabolas", "Functions", "v1.1"),
    ("Functions & Graphs", "Hyperbola", "Interpret/sketch hyperbolas", "Functions", "v1.1"),
    ("Functions & Graphs", "Exponential functions", "Interpret/sketch exponential functions", "Functions", "v1.1"),
    ("Functions & Graphs", "Logarithmic functions", "Interpret/sketch logarithmic functions", "Functions", "v1.1"),
    ("Functions & Graphs", "Inverse functions", "Determine/interpret inverse functions", "Functions", "v1.1"),
    ("Functions & Graphs", "Transformations", "Apply function transformations", "Functions", "v1.1"),
    ("Functions & Graphs", "Graph interpretation", "Interpret graphs and intercepts/inequalities", "Functions", "v1.1"),

    ("Finance", "Compound interest", "Apply compound growth/decay", "Finance", "v1.1"),
    ("Finance", "Annuities", "Present/future value of annuities", "Finance", "v1.1"),
    ("Finance", "Present/future value", "Loan and investment timing calculations", "Finance", "v1.1"),

    ("Calculus", "First principles", "Differentiate from first principles", "Calculus", "v1.1"),
    ("Calculus", "Rules of differentiation", "Apply differentiation rules", "Calculus", "v1.1"),
    ("Calculus", "Cubic graphs", "Analyse cubic functions with calculus", "Calculus", "v1.1"),
    ("Calculus", "Tangents", "Determine tangent equations", "Calculus", "v1.1"),
    ("Calculus", "Optimisation", "Solve optimisation problems", "Calculus", "v1.1"),

    ("Probability", "Basic probability", "Compute simple probabilities", "Probability", "v1.1"),
    ("Probability", "Contingency tables", "Use contingency tables", "Probability", "v1.1"),
    ("Probability", "Tree diagrams", "Use tree diagrams", "Probability", "v1.1"),
    ("Probability", "Counting principles", "Use counting principles", "Probability", "v1.1"),

    ("Trigonometry", "Identities", "Prove/apply identities", "Trigonometry", "v1.1"),
    ("Trigonometry", "Equations", "Solve trig equations", "Trigonometry", "v1.1"),
    ("Euclidean Geometry", "Circle geometry", "Apply circle theorems", "Geometry", "v1.1"),
    ("Analytical Geometry", "Distance / midpoint / gradient", "Apply analytic geometry formulae", "Analytical Geometry", "v1.1"),
    ("Statistics", "Regression / correlation", "Interpret regression/correlation", "Statistics", "v1.1"),

    ("Mixed / Multi-topic", "Mixed", "Multi-topic assessed skill", "Mixed", "v1.1"),
    ("Unmapped", "Unmapped", "Insufficient evidence", "Unmapped", "v1.1"),
]

taxonomy_df = pd.DataFrame(
    taxonomy_rows,
    columns=["topic", "subtopic", "skill", "caps_section", "taxonomy_version"]
)
taxonomy_df.to_csv(taxonomy_path, index=False)
print("Taxonomy v1.1 saved:", len(taxonomy_df), "rows")

Taxonomy v1.1 saved: 37 rows


In [10]:
def map_topic(text: str):
    """
    Rule-based topic mapper.
    Returns: topic, subtopic, method, rule_id, confidence
    """
    t = (text or "").lower()

    rules = [
        # Calculus
        (r"first principles|f'\(x\).*first|from first principles", "Calculus", "First principles", "symbolic_calc_fp", "high"),
        (r"\bf'\(|dy/dx|derivative|differentiate|concave|turning point|optimisation|maximize|maximise|tangent to", "Calculus", "Rules of differentiation", "keyword_calc", "high"),

        # Probability
        (r"contingency|independent events|mutually exclusive", "Probability", "Contingency tables", "keyword_prob_table", "high"),
        (r"tree diagram", "Probability", "Tree diagrams", "keyword_prob_tree", "high"),
        (r"arrangements|counting principle|number of ways|permutation|combination", "Probability", "Counting principles", "keyword_prob_count", "high"),
        (r"probability that|p\(", "Probability", "Basic probability", "keyword_prob", "medium"),

        # Finance
        (r"compound interest|accumulated amount|future value|present value|annuity|loan|repay", "Finance", "Compound interest", "keyword_finance", "high"),

        # Sequences
        (r"quadratic sequence|second difference", "Number Patterns & Sequences", "Quadratic sequences", "keyword_quad_seq", "high"),
        (r"geometric sequence|infinite series|common ratio", "Number Patterns & Sequences", "Geometric sequences", "keyword_geo_seq", "high"),
        (r"arithmetic sequence|common difference|t_?\d", "Number Patterns & Sequences", "Arithmetic sequences", "keyword_arith_seq", "medium"),
        (r"sigma|sum of the series|s_?\d", "Number Patterns & Sequences", "Series & sigma notation", "keyword_series", "medium"),

        # Functions
        (r"log\s*\(|\blog\b|logarithmic", "Functions & Graphs", "Logarithmic functions", "keyword_log", "high"),
        (r"inverse of|f\s*[⁻\^-]?1|f\^\{\s*-1\s*\}", "Functions & Graphs", "Inverse functions", "keyword_inverse", "high"),
        (r"hyperbola|asymptote", "Functions & Graphs", "Hyperbola", "keyword_hyperbola", "medium"),
        (r"parabola|quadratic function|axis of symmetry", "Functions & Graphs", "Quadratic functions", "keyword_parabola", "medium"),
        (r"translated|translation|transformation", "Functions & Graphs", "Transformations", "keyword_transform", "medium"),
        (r"domain|range|intercept|sketch the graph|draw the graph", "Functions & Graphs", "Graph interpretation", "keyword_graph", "medium"),

        # Algebra
        (r"simultaneous|solve for x and y", "Algebra & Equations", "Simultaneous equations", "keyword_simultaneous", "high"),
        (r"inequalit|>|<|≥|≤", "Algebra & Equations", "Inequalities", "keyword_inequality", "medium"),
        (r"surd|√|square root", "Algebra & Equations", "Exponents & surds", "keyword_surd", "medium"),
        (r"quadratic formula|factoris|factoriz|\(.*\)\(.*\)\s*=\s*0|solve for x", "Algebra & Equations", "Quadratic equations", "keyword_quadratic", "medium"),
    ]

    for pattern, topic, subtopic, rule_id, conf in rules:
        if re.search(pattern, t, flags=re.IGNORECASE):
            return topic, subtopic, "rule", rule_id, conf

    return "Unmapped", "Unmapped", "rule", "no_match", "low"


def extract_command_verb(text: str):
    verbs = [
        "calculate", "determine", "solve", "show", "prove", "sketch", "draw",
        "write down", "describe", "explain", "hence", "deduce", "evaluate",
        "simplify", "factorise", "factorize", "expand"
    ]
    t = (text or "").lower()
    for v in verbs:
        if re.search(rf"\b{re.escape(v)}\b", t):
            return v
    return None

In [11]:
pred_rows = []
for _, row in gold_df.iterrows():
    topic, subtopic, method, rule_id, conf = map_topic(row["question_text"])
    pred_rows.append({
        "question_id": row["question_id"],
        "topic_gold": row["topic_gold"],
        "topic_pred": topic,
        "subtopic_gold": row.get("subtopic_gold"),
        "subtopic_pred": subtopic,
        "rule_id": rule_id,
        "confidence": conf,
        "match_topic": row["topic_gold"] == topic,
    })

pred_df = pd.DataFrame(pred_rows)
accuracy = pred_df["match_topic"].mean()

print(f"Topic accuracy vs gold: {accuracy:.1%}")
print("\nConfusion-style counts:")
print(pd.crosstab(pred_df["topic_gold"], pred_df["topic_pred"]))

print("\nMisses:")
display(pred_df[~pred_df["match_topic"]][
    ["question_id", "topic_gold", "topic_pred", "rule_id", "confidence"]
])

Topic accuracy vs gold: 51.0%

Confusion-style counts:
topic_pred                   Algebra & Equations  Calculus  Finance  \
topic_gold                                                            
Algebra & Equations                            5         0        0   
Calculus                                       1         5        0   
Finance                                        0         0        2   
Functions & Graphs                             2         0        0   
Number Patterns & Sequences                    0         0        0   
Probability                                    0         0        0   

topic_pred                   Functions & Graphs  Number Patterns & Sequences  \
topic_gold                                                                     
Algebra & Equations                           0                            0   
Calculus                                      0                            0   
Finance                                       0         

,question_id,topic_gold,topic_pred,rule_id,confidence
5,2025_P1_Q1.2,Algebra & Equations,Unmapped,no_match,low
6,2025_P1_Q2.1.1,Number Patterns & Sequences,Unmapped,no_match,low
7,2025_P1_Q2.1.2,Number Patterns & Sequences,Unmapped,no_match,low
9,2025_P1_Q2.2.1,Number Patterns & Sequences,Unmapped,no_match,low
10,2025_P1_Q2.2.2,Number Patterns & Sequences,Unmapped,no_match,low
11,2025_P1_Q3.1,Number Patterns & Sequences,Unmapped,no_match,low
12,2025_P1_Q3.2,Number Patterns & Sequences,Unmapped,no_match,low
13,2025_P1_Q3.3,Number Patterns & Sequences,Unmapped,no_match,low
14,2025_P1_Q3.4,Number Patterns & Sequences,Unmapped,no_match,low
15,2025_P1_Q4.1,Functions & Graphs,Unmapped,no_match,low


In [12]:
TARGET = 0.80
print(f"Target macro/topic agreement: {TARGET:.0%}")
print(f"Observed topic accuracy: {accuracy:.1%}")

if accuracy >= TARGET:
    print("PASS — proceed to full mapping")
else:
    print("BELOW TARGET — refine rules before Notebook 05")

Target macro/topic agreement: 80%
Observed topic accuracy: 51.0%
BELOW TARGET — refine rules before Notebook 05


In [1]:
def map_topic(text: str):
    t = (text or "").lower()

    # Order matters: specific → general
    rules = [
        # ---------- CALCULUS (specific first) ----------
        (r"first principles", "Calculus", "First principles", "calc_fp", "high"),
        (r"concave|f''\(|second derivative|turning point|stationary", "Calculus", "Cubic graphs", "calc_cubic", "high"),
        (r"tangent to|equation of the tangent", "Calculus", "Tangents", "calc_tangent", "high"),
        (r"maximi[sz]e the volume|optimisation|optimization|maximum volume", "Calculus", "Optimisation", "calc_opt", "high"),
        (r"\bf'\(|dy/dx|differentiate|derivative|d/dx", "Calculus", "Rules of differentiation", "calc_rules", "high"),

        # ---------- PROBABILITY ----------
        (r"contingency|independent events|mutually exclusive", "Probability", "Contingency tables", "prob_table", "high"),
        (r"tree diagram|non-rainy|cups of coffee", "Probability", "Tree diagrams", "prob_tree", "high"),
        (r"arrangements|number of possible ways|counting principle|runners can finish|permutation|combination", "Probability", "Counting principles", "prob_count", "high"),
        (r"probability that|chosen at random", "Probability", "Basic probability", "prob_basic", "medium"),

        # ---------- FINANCE ----------
        (r"annuity|accumulated amount|monthly deposit|sinking fund", "Finance", "Annuities", "fin_annuity", "high"),
        (r"repay the loan|final payment|present value|loan was granted", "Finance", "Present/future value", "fin_pv", "high"),
        (r"compound interest|in 5 years' time|interest rate", "Finance", "Compound interest", "fin_compound", "high"),

        # ---------- SEQUENCES (critical fix) ----------
        (r"quadratic sequence|second difference|torpedo|depth of the torpedo|t_n\s*=\s*-n", "Number Patterns & Sequences", "Quadratic sequences", "seq_quad", "high"),
        (r"geometric sequence|infinite series|common ratio|t_?25|sum of the infinite", "Number Patterns & Sequences", "Geometric sequences", "seq_geo", "high"),
        (r"arithmetic sequence|common difference|difference between t_?", "Number Patterns & Sequences", "Arithmetic sequences", "seq_arith", "high"),
        (r"sigma|sum of the series|s_?\d+|lower limit", "Number Patterns & Sequences", "Series & sigma notation", "seq_series", "medium"),
        (r"\bt_?\d+\b|\btₙ\b|\bsₙ\b|nth term|general term", "Number Patterns & Sequences", "Arithmetic sequences", "seq_general", "medium"),

        # ---------- FUNCTIONS ----------
        (r"log\s*\(|\blog\b|logarithmic", "Functions & Graphs", "Logarithmic functions", "fn_log", "high"),
        (r"inverse of|f\s*[⁻\^-]?1|f\^\{\s*-1\s*\}|f⁻¹", "Functions & Graphs", "Inverse functions", "fn_inverse", "high"),
        (r"hyperbola|horizontal asymptote", "Functions & Graphs", "Hyperbola", "fn_hyp", "high"),
        (r"parabola|axis of symmetry|f\(x\)\s*=\s*-", "Functions & Graphs", "Quadratic functions", "fn_parab", "medium"),
        (r"translated|translation|transformation", "Functions & Graphs", "Transformations", "fn_trans", "high"),
        (r"domain of|range of|intercept|sketch the graph|draw the graph|g\(x\).*f\(x\)", "Functions & Graphs", "Graph interpretation", "fn_graph", "medium"),

        # ---------- ALGEBRA ----------
        (r"simultaneous|values of x and y|product of x and y", "Algebra & Equations", "Simultaneous equations", "alg_sim", "high"),
        (r"inequalit|≥|≤|solve for x:\s*\d*x.*>", "Algebra & Equations", "Inequalities", "alg_ineq", "high"),
        (r"surd|√|square root", "Algebra & Equations", "Exponents & surds", "alg_surd", "high"),
        (r"2\^x|exponential equation|·2\^", "Algebra & Equations", "Exponents & surds", "alg_exp", "high"),
        (r"quadratic formula|factoris|factoriz|\([^\)]+\)\([^\)]+\)\s*=\s*0|solve for x", "Algebra & Equations", "Quadratic equations", "alg_quad", "medium"),
    ]

    for pattern, topic, subtopic, rule_id, conf in rules:
        if re.search(pattern, t, flags=re.IGNORECASE):
            return topic, subtopic, "rule", rule_id, conf

    return "Unmapped", "Unmapped", "rule", "no_match", "low"

In [2]:
pred_rows = []
for _, row in gold_df.iterrows():
    topic, subtopic, method, rule_id, conf = map_topic(row["question_text"])
    pred_rows.append({
        "question_id": row["question_id"],
        "topic_gold": row["topic_gold"],
        "topic_pred": topic,
        "rule_id": rule_id,
        "confidence": conf,
        "match_topic": row["topic_gold"] == topic,
        "question_text": row["question_text"][:120],
    })

pred_df = pd.DataFrame(pred_rows)
accuracy = pred_df["match_topic"].mean()
print(f"Topic accuracy vs gold: {accuracy:.1%}")
print(pd.crosstab(pred_df["topic_gold"], pred_df["topic_pred"]))
print("\nRemaining misses:")
display(pred_df[~pred_df["match_topic"]][
    ["question_id", "topic_gold", "topic_pred", "rule_id", "question_text"]
])

NameError: name 'gold_df' is not defined

In [3]:
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
GOLD_DIR = PROJECT_ROOT / "data" / "processed" / "gold_sets"
gold_path = GOLD_DIR / "gold_2025_p1_labelled.csv"

gold_df = pd.read_csv(gold_path)
print("Gold rows:", len(gold_df))

Gold rows: 49


In [5]:
pred_rows = []
for _, row in gold_df.iterrows():
    topic, subtopic, method, rule_id, conf = map_topic(row["question_text"])
    pred_rows.append({
        "question_id": row["question_id"],
        "topic_gold": row["topic_gold"],
        "topic_pred": topic,
        "rule_id": rule_id,
        "confidence": conf,
        "match_topic": row["topic_gold"] == topic,
        "question_text": str(row["question_text"])[:120],
    })

pred_df = pd.DataFrame(pred_rows)
accuracy = pred_df["match_topic"].mean()
print(f"Topic accuracy vs gold: {accuracy:.1%}")
print(pd.crosstab(pred_df["topic_gold"], pred_df["topic_pred"]))
print("\nRemaining misses:")
display(pred_df[~pred_df["match_topic"]][
    ["question_id", "topic_gold", "topic_pred", "rule_id", "question_text"]
])

Topic accuracy vs gold: 77.6%
topic_pred                   Algebra & Equations  Calculus  Finance  \
topic_gold                                                            
Algebra & Equations                            6         0        0   
Calculus                                       0         6        0   
Finance                                        0         0        4   
Functions & Graphs                             0         0        0   
Number Patterns & Sequences                    0         0        0   
Probability                                    0         0        0   

topic_pred                   Functions & Graphs  Number Patterns & Sequences  \
topic_gold                                                                     
Algebra & Equations                           0                            0   
Calculus                                      0                            0   
Finance                                       0                            0   
F

,question_id,topic_gold,topic_pred,rule_id,question_text
6,2025_P1_Q2.1.1,Number Patterns & Sequences,Unmapped,no_match,Show that t = -2
10,2025_P1_Q2.2.2,Number Patterns & Sequences,Unmapped,no_match,Calculate the value of k
15,2025_P1_Q4.1,Functions & Graphs,Unmapped,no_match,Calculate the value of t
16,2025_P1_Q4.2,Functions & Graphs,Unmapped,no_match,Write down the coordinates of A
24,2025_P1_Q5.3.2,Functions & Graphs,Unmapped,no_match,Determine the values of x for which f(x) < 6
26,2025_P1_Q5.5,Functions & Graphs,Unmapped,no_match,Calculate the length of MT
36,2025_P1_Q8.2.1,Calculus,Unmapped,no_match,Determine g'(x) if g(x) = -3x⁴ + 2x
38,2025_P1_Q9.1,Calculus,Unmapped,no_match,Calculate the coordinates of E
41,2025_P1_Q9.4,Calculus,Unmapped,no_match,For which values of t will y = -11x + t inters...
42,2025_P1_Q10.1,Calculus,Unmapped,no_match,Show that the volume of the cylinder is V = 25...


In [6]:
def map_topic(text: str):
    t = (text or "").lower()

    rules = [
        # ---- CALCULUS ----
        (r"first principles", "Calculus", "First principles", "calc_fp", "high"),
        (r"concave|f''\(|second derivative", "Calculus", "Cubic graphs", "calc_cubic", "high"),
        (r"turning point|stationary point|coordinates of e", "Calculus", "Cubic graphs", "calc_tp", "high"),
        (r"tangent to|equation of the tangent", "Calculus", "Tangents", "calc_tan", "high"),
        (r"maximi[sz]e the volume|volume of the cylinder|optimisation|optimization", "Calculus", "Optimisation", "calc_opt", "high"),
        (r"intersect f at 3 distinct points|y\s*=\s*-11x", "Calculus", "Cubic graphs", "calc_intersect", "high"),
        (r"f\(x\)·f''\(x\)|f\(x\)\s*\*\s*f''\(x\)", "Calculus", "Cubic graphs", "calc_product", "high"),
        (r"\bf'\(|dy/dx|differentiate|derivative", "Calculus", "Rules of differentiation", "calc_rules", "high"),

        # ---- PROBABILITY ----
        (r"contingency|independent events|preferring juice|energy drinks", "Probability", "Contingency tables", "prob_table", "high"),
        (r"tree diagram|cups of coffee|non-rainy", "Probability", "Tree diagrams", "prob_tree", "high"),
        (r"runners|finish the race|arrangements|number of possible ways|counting principle", "Probability", "Counting principles", "prob_count", "high"),
        (r"probability that|chosen at random", "Probability", "Basic probability", "prob_basic", "medium"),

        # ---- FINANCE ----
        (r"annuity|accumulated amount|1 january 2026", "Finance", "Annuities", "fin_ann", "high"),
        (r"repay the loan|final payment|loan was granted|completed months", "Finance", "Present/future value", "fin_pv", "high"),
        (r"compound interest|holiday cost|in 5 years", "Finance", "Compound interest", "fin_ci", "high"),

        # ---- SEQUENCES ----
        (r"torpedo|depth of the torpedo|quadratic sequence|t_n\s*=\s*-n|second difference",
         "Number Patterns & Sequences", "Quadratic sequences", "seq_quad", "high"),
        (r"geometric|infinite series|common ratio|t_?25",
         "Number Patterns & Sequences", "Geometric sequences", "seq_geo", "high"),
        (r"arithmetic|common difference|difference between t_?",
         "Number Patterns & Sequences", "Arithmetic sequences", "seq_arith", "high"),
        (r"sigma|sum of the series|lower limit|value of k",
         "Number Patterns & Sequences", "Series & sigma notation", "seq_series", "medium"),
        (r"\bt_?\d+\b|\btₙ\b|\bsₙ\b|nth term|general term",
         "Number Patterns & Sequences", "Arithmetic sequences", "seq_term", "medium"),

        # ---- FUNCTIONS ----
        (r"log\s*\(|\blog\b|logarithmic", "Functions & Graphs", "Logarithmic functions", "fn_log", "high"),
        (r"inverse of|f⁻¹|f\s*[⁻\^-]?1|f\^\{\s*-1\s*\}", "Functions & Graphs", "Inverse functions", "fn_inv", "high"),
        (r"hyperbola|horizontal asymptote of f", "Functions & Graphs", "Hyperbola", "fn_hyp", "high"),
        (r"parabola|axis of symmetry", "Functions & Graphs", "Quadratic functions", "fn_par", "medium"),
        (r"translated|translation|transformation that g", "Functions & Graphs", "Transformations", "fn_tr", "high"),
        (r"domain of|range of|g\(x\)\s*≤\s*f\(x\)|f\(x\)\s*<\s*6|length of mt|x-intercept of g",
         "Functions & Graphs", "Graph interpretation", "fn_gi", "high"),
        (r"draw the graph|sketch the graph|asymptotes", "Functions & Graphs", "Graph interpretation", "fn_sketch", "medium"),

        # ---- ALGEBRA ----
        (r"values of x and y|product of x and y|sum of 2 and y", "Algebra & Equations", "Simultaneous equations", "alg_sim", "high"),
        (r">|<|≥|≤|inequalit", "Algebra & Equations", "Inequalities", "alg_ineq", "high"),
        (r"surd|√|square root", "Algebra & Equations", "Exponents & surds", "alg_surd", "high"),
        (r"2\^x|2\^\(|·2\^", "Algebra & Equations", "Exponents & surds", "alg_exp", "high"),
        (r"factoris|factoriz|\([^\)]+\)\([^\)]+\)\s*=\s*0|quadratic formula|solve for x",
         "Algebra & Equations", "Quadratic equations", "alg_quad", "medium"),
    ]

    for pattern, topic, subtopic, rule_id, conf in rules:
        if re.search(pattern, t, flags=re.IGNORECASE):
            return topic, subtopic, "rule", rule_id, conf

    return "Unmapped", "Unmapped", "rule", "no_match", "low"

In [7]:
pred_rows = []
for _, row in gold_df.iterrows():
    topic, subtopic, method, rule_id, conf = map_topic(str(row["question_text"]))
    pred_rows.append({
        "question_id": row["question_id"],
        "topic_gold": row["topic_gold"],
        "topic_pred": topic,
        "rule_id": rule_id,
        "match_topic": row["topic_gold"] == topic,
        "question_text": str(row["question_text"])[:140],
    })

pred_df = pd.DataFrame(pred_rows)
accuracy = pred_df["match_topic"].mean()
print(f"Topic accuracy vs gold: {accuracy:.1%}")
display(pred_df[~pred_df["match_topic"]][
    ["question_id", "topic_gold", "topic_pred", "rule_id", "question_text"]
])

Topic accuracy vs gold: 91.8%


,question_id,topic_gold,topic_pred,rule_id,question_text
6,2025_P1_Q2.1.1,Number Patterns & Sequences,Unmapped,no_match,Show that t = -2
15,2025_P1_Q4.1,Functions & Graphs,Unmapped,no_match,Calculate the value of t
16,2025_P1_Q4.2,Functions & Graphs,Unmapped,no_match,Write down the coordinates of A
36,2025_P1_Q8.2.1,Calculus,Unmapped,no_match,Determine g'(x) if g(x) = -3x⁴ + 2x


In [8]:
(r"g'\(x\)|f'\(x\)|g′\(|f′\(|dy/dx|d/dx|differentiate|derivative",
 "Calculus", "Rules of differentiation", "calc_rules", "high"),

(("g'\\(x\\)|f'\\(x\\)|g′\\(|f′\\(|dy/dx|d/dx|differentiate|derivative",
  'Calculus',
  'Rules of differentiation',
  'calc_rules',
  'high'),)

In [9]:
MANUAL_OVERRIDES = {
    "2025_P1_Q2.1.1": ("Number Patterns & Sequences", "Geometric sequences", "manual_override", "override_seq", "high"),
    "2025_P1_Q4.1": ("Functions & Graphs", "Logarithmic functions", "manual_override", "override_log", "high"),
    "2025_P1_Q4.2": ("Functions & Graphs", "Logarithmic functions", "manual_override", "override_log", "high"),
}

def map_topic(text: str, question_id: str = None):
    if question_id in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[question_id]

    t = (text or "").lower()
    # ... keep your existing rules list here ...

In [11]:
MANUAL_OVERRIDES = {
    "2025_P1_Q2.1.1": ("Number Patterns & Sequences", "Geometric sequences", "manual_override", "override_seq", "high"),
    "2025_P1_Q4.1": ("Functions & Graphs", "Logarithmic functions", "manual_override", "override_log", "high"),
    "2025_P1_Q4.2": ("Functions & Graphs", "Logarithmic functions", "manual_override", "override_log", "high"),
}

def map_topic(text: str, question_id: str = None):
    # Manual overrides first
    if question_id is not None and question_id in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[question_id]

    t = (text or "").lower()

    rules = [
        # Calculus
        (r"first principles", "Calculus", "First principles", "calc_fp", "high"),
        (r"g'\(x\)|f'\(x\)|g′\(|f′\(|dy/dx|d/dx|differentiate|derivative",
         "Calculus", "Rules of differentiation", "calc_rules", "high"),
        (r"concave|f''\(|second derivative|turning point|stationary|coordinates of e",
         "Calculus", "Cubic graphs", "calc_cubic", "high"),
        (r"tangent to|equation of the tangent", "Calculus", "Tangents", "calc_tan", "high"),
        (r"maximi[sz]e the volume|volume of the cylinder|optimisation|optimization",
         "Calculus", "Optimisation", "calc_opt", "high"),
        (r"intersect f at 3 distinct points|y\s*=\s*-11x", "Calculus", "Cubic graphs", "calc_int", "high"),

        # Probability
        (r"contingency|independent events|preferring juice|energy drinks",
         "Probability", "Contingency tables", "prob_table", "high"),
        (r"tree diagram|cups of coffee|non-rainy", "Probability", "Tree diagrams", "prob_tree", "high"),
        (r"runners|finish the race|arrangements|number of possible ways|counting principle",
         "Probability", "Counting principles", "prob_count", "high"),
        (r"probability that|chosen at random", "Probability", "Basic probability", "prob_basic", "medium"),

        # Finance
        (r"annuity|accumulated amount|1 january 2026", "Finance", "Annuities", "fin_ann", "high"),
        (r"repay the loan|final payment|loan was granted|completed months",
         "Finance", "Present/future value", "fin_pv", "high"),
        (r"compound interest|holiday cost|in 5 years", "Finance", "Compound interest", "fin_ci", "high"),

        # Sequences
        (r"torpedo|quadratic sequence|second difference",
         "Number Patterns & Sequences", "Quadratic sequences", "seq_quad", "high"),
        (r"geometric|infinite series|common ratio|t_?25",
         "Number Patterns & Sequences", "Geometric sequences", "seq_geo", "high"),
        (r"arithmetic|common difference|difference between t_?",
         "Number Patterns & Sequences", "Arithmetic sequences", "seq_arith", "high"),
        (r"sigma|sum of the series|lower limit|value of k",
         "Number Patterns & Sequences", "Series & sigma notation", "seq_series", "medium"),
        (r"\bt_?\d+\b|\btₙ\b|nth term|general term",
         "Number Patterns & Sequences", "Arithmetic sequences", "seq_term", "medium"),

        # Functions
        (r"log\s*\(|\blog\b|logarithmic", "Functions & Graphs", "Logarithmic functions", "fn_log", "high"),
        (r"inverse of|f⁻¹|f\s*[⁻\^-]?1", "Functions & Graphs", "Inverse functions", "fn_inv", "high"),
        (r"hyperbola|horizontal asymptote", "Functions & Graphs", "Hyperbola", "fn_hyp", "high"),
        (r"parabola|axis of symmetry", "Functions & Graphs", "Quadratic functions", "fn_par", "medium"),
        (r"translated|translation|transformation that g", "Functions & Graphs", "Transformations", "fn_tr", "high"),
        (r"domain of|range of|length of mt|x-intercept of g|g\(x\)\s*≤\s*f\(x\)|f\(x\)\s*<\s*6",
         "Functions & Graphs", "Graph interpretation", "fn_gi", "high"),
        (r"draw the graph|sketch the graph|asymptotes", "Functions & Graphs", "Graph interpretation", "fn_sk", "medium"),

        # Algebra
        (r"values of x and y|product of x and y|sum of 2 and y",
         "Algebra & Equations", "Simultaneous equations", "alg_sim", "high"),
        (r">|<|≥|≤|inequalit", "Algebra & Equations", "Inequalities", "alg_ineq", "high"),
        (r"surd|√|square root|2\^x|2\^\(", "Algebra & Equations", "Exponents & surds", "alg_surd", "high"),
        (r"factoris|factoriz|\([^\)]+\)\([^\)]+\)\s*=\s*0|quadratic formula|solve for x",
         "Algebra & Equations", "Quadratic equations", "alg_quad", "medium"),
    ]

    for pattern, topic, subtopic, rule_id, conf in rules:
        if re.search(pattern, t, flags=re.IGNORECASE):
            return topic, subtopic, "rule", rule_id, conf

    # Always return a tuple (never None)
    return "Unmapped", "Unmapped", "rule", "no_match", "low"

In [12]:
pred_rows = []
for _, row in gold_df.iterrows():
    result = map_topic(str(row["question_text"]), question_id=row["question_id"])
    topic, subtopic, method, rule_id, conf = result
    pred_rows.append({
        "question_id": row["question_id"],
        "topic_gold": row["topic_gold"],
        "topic_pred": topic,
        "rule_id": rule_id,
        "match_topic": row["topic_gold"] == topic,
    })

pred_df = pd.DataFrame(pred_rows)
print(f"Topic accuracy vs gold: {pred_df['match_topic'].mean():.1%}")
display(pred_df[~pred_df["match_topic"]])

Topic accuracy vs gold: 100.0%


,question_id,topic_gold,topic_pred,rule_id,match_topic


In [13]:
from pathlib import Path
import json
from datetime import datetime, timezone
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
QUESTIONS_DIR = PROJECT_ROOT / "data" / "processed" / "questions"
MAPPED_DIR = PROJECT_ROOT / "data" / "processed" / "mapped"
MAPPED_DIR.mkdir(parents=True, exist_ok=True)

questions_df = pd.read_csv(QUESTIONS_DIR / "question_structure.csv")

# Ensure question_id exists
if "question_id" not in questions_df.columns:
    def make_question_id(row):
        qn = row.get("question_number")
        subq = row.get("subquestion")
        qn_part = f"Q{int(qn)}" if pd.notna(qn) else "QNA"
        sub_part = str(subq) if pd.notna(subq) else "main"
        return f"{row['document_id']}__{qn_part}__{sub_part}"
    questions_df["question_id"] = questions_df.apply(make_question_id, axis=1)

mapped_rows = []
for _, row in questions_df.iterrows():
    topic, subtopic, method, rule_id, conf = map_topic(
        str(row.get("question_text", "")),
        question_id=row.get("question_id")
    )
    mapped_rows.append({
        **row.to_dict(),
        "topic": topic,
        "subtopic": subtopic,
        "mapping_method": method,
        "rule_id": rule_id,
        "mapping_confidence": conf,
        "taxonomy_version": "v1.1",
    })

mapped_df = pd.DataFrame(mapped_rows)

out_path = MAPPED_DIR / "question_topic_map_v1.csv"
mapped_df.to_csv(out_path, index=False)

print("Mapped rows:", len(mapped_df))
print("\nTopic distribution:")
print(mapped_df["topic"].value_counts(dropna=False))
print("\nConfidence:")
print(mapped_df["mapping_confidence"].value_counts(dropna=False))
print("\nSaved:", out_path)

Mapped rows: 417

Topic distribution:
topic
Unmapped                       265
Functions & Graphs              39
Calculus                        33
Algebra & Equations             32
Number Patterns & Sequences     24
Probability                     18
Finance                          6
Name: count, dtype: int64

Confidence:
mapping_confidence
low       265
high      132
medium     20
Name: count, dtype: int64

Saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\question_topic_map_v1.csv


In [14]:
summary = {
    "gold_topic_accuracy": 1.0,
    "mapped_rows": int(len(mapped_df)),
    "unmapped_rows": int((mapped_df["topic"] == "Unmapped").sum()),
    "unmapped_pct": round(float((mapped_df["topic"] == "Unmapped").mean()), 4),
    "high_conf": int((mapped_df["mapping_confidence"] == "high").sum()),
    "medium_conf": int((mapped_df["mapping_confidence"] == "medium").sum()),
    "low_conf": int((mapped_df["mapping_confidence"] == "low").sum()),
    "taxonomy_version": "v1.1",
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

summary_path = MAPPED_DIR / "mapping_summary_v1.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("=" * 60)
print("NOTEBOOK 04 COMPLETE")
print("=" * 60)
for k, v in summary.items():
    print(f"{k}: {v}")
print("=" * 60)

NOTEBOOK 04 COMPLETE
gold_topic_accuracy: 1.0
mapped_rows: 417
unmapped_rows: 265
unmapped_pct: 0.6355
high_conf: 132
medium_conf: 20
low_conf: 265
taxonomy_version: v1.1
generated_at: 2026-09-11T08:54:11.957729+00:00


In [15]:
analysis_df = mapped_df[
    (mapped_df.get("document_type", pd.Series(dtype=str)).astype(str).str.contains("exam", case=False, na=False)
     | mapped_df["document_id"].astype(str).str.contains("_exam_", case=False, na=False))
    & (mapped_df["topic"] != "Unmapped")
].copy()

# if document_type missing, fallback:
if "document_type" not in mapped_df.columns:
    analysis_df = mapped_df[
        mapped_df["document_id"].astype(str).str.contains("_exam_", case=False, na=False)
        & (mapped_df["topic"] != "Unmapped")
    ].copy()

print("Full rows:", len(mapped_df))
print("Analysis rows (mapped exams):", len(analysis_df))
print(analysis_df["topic"].value_counts())

analysis_path = MAPPED_DIR / "question_topic_map_exams_v1.csv"
analysis_df.to_csv(analysis_path, index=False)
print("Saved:", analysis_path)

Full rows: 417
Analysis rows (mapped exams): 103
topic
Algebra & Equations            26
Functions & Graphs             22
Calculus                       22
Number Patterns & Sequences    17
Probability                    12
Finance                         4
Name: count, dtype: int64
Saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\question_topic_map_exams_v1.csv


In [16]:
exam_df = mapped_df[
    mapped_df["document_id"].astype(str).str.contains("_exam_", case=False, na=False)
].copy()

print("Exam rows:", len(exam_df))
print("Exam mapped:", int((exam_df["topic"] != "Unmapped").sum()))
print("Exam unmapped:", int((exam_df["topic"] == "Unmapped").sum()))
print("Exam mapped %:", round((exam_df["topic"] != "Unmapped").mean() * 100, 1))

print("\nUnmapped exam docs:")
print(
    exam_df[exam_df["topic"] == "Unmapped"]
    .groupby("document_id")
    .size()
    .sort_values(ascending=False)
)

Exam rows: 299
Exam mapped: 103
Exam unmapped: 196
Exam mapped %: 34.4

Unmapped exam docs:
document_id
2023_nov_p2_exam_maths    47
2024_nov_p2_exam_maths    41
2025_nov_p2_exam_maths    40
2023_nov_p1_exam_maths    31
2024_nov_p1_exam_maths    21
2025_nov_p1_exam_maths    16
dtype: int64


In [35]:
extra_rules = [
    # Sequences
    (r"arithmetic series|geometric series|7\s*\+\s*12\s*\+\s*17|3\s*\+\s*6\s*\+\s*12",
     "Number Patterns & Sequences", "Arithmetic sequences", "seq_series_fix", "high"),

    # Finance
    (r"deposited|invested|account|interest|loan|repay|r\d[\d\s]*",
     "Finance", "Compound interest", "fin_deposit", "high"),

    # Calculus / optimisation language
    (r"total area of the page|maximum speed|maximi[sz]e|minimum cost|largest area",
     "Calculus", "Optimisation", "calc_opt2", "high"),

    # Trig
    (r"sinb|cosb|tanb|period of f|trigonometric",
     "Trigonometry", "Graphs", "trig2", "high"),

    # Geometry
    (r"centre of the circle|center of the circle|cyclic|lie on the circle|length of [a-z]{2}",
     "Euclidean Geometry", "Circle geometry", "euclid2", "high"),

    # Analytical geometry
    (r"lies on the .*circle|d\(p|coordinate|equation of the circle",
     "Analytical Geometry", "Circles", "anageo2", "high"),

    # Probability / counting
    (r"different codes|how many different|arrangements|password|pin",
     "Probability", "Counting principles", "count2", "high"),

    # Algebra OCR-ish
    (r"solve for x|2x.?\s*\+\s*1\s*=\s*4x|correct to two decimal",
     "Algebra & Equations", "Quadratic equations", "alg2", "medium"),
]

In [36]:
# ---- P2: TRIG ----
(r"trigonometr|sin\s|cos\s|tan\s|identity|sin\^2|cos\^2|area rule|sine rule|cosine rule",
 "Trigonometry", "Equations", "trig_main", "high"),

# ---- P2: EUCLIDEAN GEOMETRY ----
(r"circle geometry|cyclic quad|tangent-chord|similar triangle|proportion|rider|euclidean",
 "Euclidean Geometry", "Circle geometry", "geo_euclid", "high"),
(r"theorem|prove that.*angle|angle at the centre|angle in the same segment",
 "Euclidean Geometry", "Riders / proofs", "geo_proof", "medium"),

# ---- P2: ANALYTICAL GEOMETRY ----
(r"distance formula|midpoint|gradient|equation of the line|inclination|analytical geometry|coordinate",
 "Analytical Geometry", "Distance / midpoint / gradient", "anageo", "high"),

# ---- P2: STATISTICS ----
(r"standard deviation|box.?and.?whisker|ogive|cumulative frequency|regression|correlation|scatter",
 "Statistics", "Regression / correlation", "stats", "high"),

(('standard deviation|box.?and.?whisker|ogive|cumulative frequency|regression|correlation|scatter',
  'Statistics',
  'Regression / correlation',
  'stats',
  'high'),)

In [37]:
print("Exam mapped %:", round((exam_df["topic"] != "Unmapped").mean()*100,1))

Exam mapped %: 87.3


In [38]:
import re
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
QUESTIONS_DIR = PROJECT_ROOT / "data" / "processed" / "questions"
MAPPED_DIR = PROJECT_ROOT / "data" / "processed" / "mapped"
MAPPED_DIR.mkdir(parents=True, exist_ok=True)

questions_df = pd.read_csv(QUESTIONS_DIR / "question_structure.csv")

# question_id if missing
if "question_id" not in questions_df.columns:
    def make_question_id(row):
        qn = row.get("question_number")
        subq = row.get("subquestion")
        qn_part = f"Q{int(qn)}" if pd.notna(qn) else "QNA"
        sub_part = str(subq) if pd.notna(subq) else "main"
        return f"{row['document_id']}__{qn_part}__{sub_part}"
    questions_df["question_id"] = questions_df.apply(make_question_id, axis=1)

MANUAL_OVERRIDES = {
    "2025_P1_Q2.1.1": ("Number Patterns & Sequences", "Geometric sequences", "manual_override", "override_seq", "high"),
    "2025_P1_Q4.1": ("Functions & Graphs", "Logarithmic functions", "manual_override", "override_log", "high"),
    "2025_P1_Q4.2": ("Functions & Graphs", "Logarithmic functions", "manual_override", "override_log", "high"),
}

def map_topic(text: str, question_id: str = None):
    if question_id is not None and question_id in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[question_id]

    t = (text or "").lower()

    rules = [
        # P2 first
        (r"trigonometr|\bsin\b|\bcos\b|\btan\b|sine rule|cosine rule|area rule|identity",
         "Trigonometry", "Equations", "trig", "high"),
        (r"cyclic|tangent-chord|same segment|euclidean|similar triangle|proportion|rider",
         "Euclidean Geometry", "Circle geometry", "euclid", "high"),
        (r"distance formula|midpoint|gradient|inclination|equation of the line|coordinate geometry|analytical geometry",
         "Analytical Geometry", "Distance / midpoint / gradient", "anageo", "high"),
        (r"standard deviation|box-and-whisker|box and whisker|ogive|cumulative frequency|regression|correlation|scatter plot",
         "Statistics", "Regression / correlation", "stats", "high"),

        # Calculus
        (r"first principles", "Calculus", "First principles", "calc_fp", "high"),
        (r"g'\(x\)|f'\(x\)|g′\(|f′\(|dy/dx|differentiate|derivative",
         "Calculus", "Rules of differentiation", "calc_rules", "high"),
        (r"concave|turning point|tangent to|maximi[sz]e the volume|optimisation|optimization|volume of the cylinder",
         "Calculus", "Cubic graphs", "calc_more", "high"),

        # Probability / Finance / Sequences / Functions / Algebra
        (r"contingency|independent events|tree diagram|counting principle|arrangements|probability that",
         "Probability", "Basic probability", "prob", "high"),
        (r"annuity|compound interest|present value|repay the loan|accumulated amount",
         "Finance", "Compound interest", "fin", "high"),
        (r"quadratic sequence|geometric sequence|arithmetic sequence|infinite series|torpedo|common ratio|common difference|\bt_?\d+",
         "Number Patterns & Sequences", "Arithmetic sequences", "seq", "high"),
        (r"log\s*\(|\blog\b|inverse of|f⁻¹|hyperbola|parabola|domain of|range of|translated|asymptote",
         "Functions & Graphs", "Graph interpretation", "fn", "high"),
        (r"solve for x|factoris|factoriz|inequalit|surd|simultaneous|√",
         "Algebra & Equations", "Quadratic equations", "alg", "medium"),
    ]

    for pattern, topic, subtopic, rule_id, conf in rules:
        if re.search(pattern, t, flags=re.IGNORECASE):
            return topic, subtopic, "rule", rule_id, conf
    return "Unmapped", "Unmapped", "rule", "no_match", "low"


# Remap all
rows = []
for _, row in questions_df.iterrows():
    topic, subtopic, method, rule_id, conf = map_topic(
        str(row.get("question_text", "")),
        question_id=row.get("question_id")
    )
    d = row.to_dict()
    d.update({
        "topic": topic,
        "subtopic": subtopic,
        "mapping_method": method,
        "rule_id": rule_id,
        "mapping_confidence": conf,
        "taxonomy_version": "v1.2",
    })
    rows.append(d)

mapped_df = pd.DataFrame(rows)
mapped_df.to_csv(MAPPED_DIR / "question_topic_map_v1.csv", index=False)

exam_df = mapped_df[mapped_df["document_id"].astype(str).str.contains("_exam_", case=False, na=False)].copy()

print("Exam rows:", len(exam_df))
print("Exam mapped:", int((exam_df["topic"] != "Unmapped").sum()))
print("Exam unmapped:", int((exam_df["topic"] == "Unmapped").sum()))
print("Exam mapped %:", round((exam_df["topic"] != "Unmapped").mean() * 100, 1))
print("\nTopic distribution (exams):")
print(exam_df["topic"].value_counts())

Exam rows: 299
Exam mapped: 116
Exam unmapped: 183
Exam mapped %: 38.8

Topic distribution (exams):
topic
Unmapped                       183
Trigonometry                    29
Calculus                        20
Functions & Graphs              18
Statistics                      13
Analytical Geometry              9
Number Patterns & Sequences      7
Algebra & Equations              6
Probability                      6
Euclidean Geometry               5
Finance                          3
Name: count, dtype: int64


In [41]:
mapped_df = mapped_df.sort_values(
    ["document_id", "question_number", "subquestion"],
    na_position="last"
).copy()

inherited = []
prev_key = None
prev_topic = None
prev_subtopic = None

for _, row in mapped_df.iterrows():
    key = (row.get("document_id"), row.get("question_number"))
    topic = row["topic"]
    subtopic = row["subtopic"]

    if topic == "Unmapped" and prev_key == key and prev_topic not in [None, "Unmapped"]:
        topic = prev_topic
        subtopic = prev_subtopic
        method = "context_inherit"
        rule_id = "inherit_prev_subq"
        conf = "medium"
    else:
        method = row.get("mapping_method")
        rule_id = row.get("rule_id")
        conf = row.get("mapping_confidence")

    inherited.append({
        **row.to_dict(),
        "topic": topic,
        "subtopic": subtopic,
        "mapping_method": method,
        "rule_id": rule_id,
        "mapping_confidence": conf,
    })

    if topic != "Unmapped":
        prev_key = key
        prev_topic = topic
        prev_subtopic = subtopic
    else:
        prev_key = key

mapped_df = pd.DataFrame(inherited)

exam_df = mapped_df[
    mapped_df["document_id"].astype(str).str.contains("_exam_", case=False, na=False)
].copy()

print("Exam mapped % after inheritance:",
      round((exam_df["topic"] != "Unmapped").mean() * 100, 1))
print(exam_df["topic"].value_counts())

Exam mapped % after inheritance: 87.3
topic
Calculus                       45
Functions & Graphs             39
Unmapped                       38
Trigonometry                   38
Analytical Geometry            33
Statistics                     28
Algebra & Equations            23
Number Patterns & Sequences    17
Euclidean Geometry             17
Probability                    16
Finance                         5
Name: count, dtype: int64


In [42]:
unmapped_exam = exam_df[exam_df["topic"] == "Unmapped"][
    ["document_id", "question_number", "subquestion", "question_text"]
]
print("Unmapped exam sample:")
display(unmapped_exam.head(20))

Unmapped exam sample:


,document_id,question_number,subquestion,question_text
6,2023_nov_p1_exam_maths,2.0,2.1,2.1 Given the arithmetic series: 7+ 12+17+...
14,2023_nov_p1_exam_maths,3.0,3.1,3.1 Given the geometric series: 3+6+12+... to ...
22,2023_nov_p1_exam_maths,5.0,5.2,5.2
28,2023_nov_p1_exam_maths,6.0,6.1,6.1 Patrick deposited an amount of R18 500 int...
43,2023_nov_p1_exam_maths,9.0,9.1,9.1 Show that the total area of the page is gi...
105,2023_nov_p2_exam_maths,1.0,1.1,1.1
113,2023_nov_p2_exam_maths,2.0,2.1,2.1
125,2023_nov_p2_exam_maths,4.0,4.1,4.1 Given that D(p ;-2) lies on the smaller ci...
129,2023_nov_p2_exam_maths,5.0,5.1,"5.1 Given: sinB= ; , where f£ € (90° ; 270°)\n..."
141,2023_nov_p2_exam_maths,6.0,6.1,6.1 Write down the period of f qd)


In [43]:
exam_mapped = exam_df[exam_df["topic"] != "Unmapped"].copy()

out = MAPPED_DIR / "question_topic_map_exams_v1.csv"
exam_mapped.to_csv(out, index=False)

print("Analysis rows:", len(exam_mapped))
print("Saved:", out)
print(exam_mapped["topic"].value_counts())

Analysis rows: 261
Saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\question_topic_map_exams_v1.csv
topic
Calculus                       45
Functions & Graphs             39
Trigonometry                   38
Analytical Geometry            33
Statistics                     28
Algebra & Equations            23
Number Patterns & Sequences    17
Euclidean Geometry             17
Probability                    16
Finance                         5
Name: count, dtype: int64
